In [1]:
# Extending Built-in Object Types
    # customizing built-in types
    # classes vs. types
    # __slots__ and properties
    # static methods and class methods
    # decorators and metaclasses
    # the complete super() mechanism
    # common OOP “gotchas

    # Embedding/delegation — put a built-in object such as a list inside your class and delegate operations to it.
    # Inheritance — directly subclass a built-in type such as list

# Extending Types by Embedding
from setwrapper import Set
x = Set([1, 3, 5, 7, 3])
x.union(Set([1, 4, 7]))
x | Set([1, 4, 6, 4])

# Extending Types by Subclassing
    # Instead of embedding a built-in type inside your class, you can inherit directly from it.

    # | Embedding                    | Subclassing                                 |
    # | ---------------------------- | ------------------------------------------- |
    # | Class **contains** a list    | Class **is a** list                         |
    # | `self.data` stores the list  | `self` is the list                          |
    # | You must delegate operations | Most operations are inherited automatically |
    # | More code                    | Usually shorter                             |

%run typesubclass.py
%run setsubclass.py
    # Subclassing = take an existing built-in type and customize or extend it, while keeping its existing behavior.

# The Python Object Model
    # In Python, classes are types, and classes themselves are objects.

# Classes Are Types Are Classes
class Hack: pass                    # A humble user-defined class
I = Hack()                          # Make an instance by calling the class
type(I)                             # Type is the user-defined class of origin
type(Hack)                          # User-defined classes are instances of type
I.__class__, Hack.__class__
isinstance(I, object), isinstance(Hack, object)
I = 'hack'                          # Make an instance by literal syntax, or str()
type(I)                             # Built-in objects are instances of classes
type(str)                           # Built-in classes are instances of type
I.__class__, str.__class__
isinstance(I, object), isinstance(str, object)

type(type)                          # The type class ends the chain
type(type(type))                    # Hmm...the top of the chain

# Some Instances Are More Equal Than Others
    # Python has two kinds of instances: class objects and ordinary objects. Both are objects, but they behave differently

class C:                    # C is itself a class object, created by the metaclass type.
    pass
I = C()                     # I is an ordinary instance created from class C

print(isinstance(I, type))  # False
print(isinstance(C, type))  # True

    # A metaclass is a class that creates classes. Python's default metaclass is type.
    # Here, "type" creates class C, and calling C() creates instance I.

# The Inheritance Bifurcation
    # Python has two inheritance search paths: one for ordinary instances and another, additional path for class objects.

# 1. Ordinary instance inheritance
isinstance(I, C), type(I)
I, I.__class__, I.__class__.__bases__
C, C.__bases__, C.__class__, C.__class__.__bases__
# The search path is: I → C → object

# 2. Class inheritance has an additional search
I, I.__class__.__mro__
C.__mro__, C.__class__.__mro__
[I.__dict__] + [x.__dict__ for x in I.__class__.__mro__]
[x.__dict__ for x in C.__mro__] + [x.__dict__ for x in C.__class__.__mro__]

# So the class-object search involves two MRO sequences: C → object, then type → object as the additional search path.

# 3. Why does C.__bases__ work but I.__bases__ fail?
print(C.__bases__)  # (<class 'object'>,)
print(I.__bases__)  # AttributeError

'__bases__' in I.__dict__           # Not in instance
'__bases__' in C.__dict__           # Not in instance's class
'__bases__' in C.__class__.__dict__ # In instance's class's class
C.__bases__ # Instance does not inherit!
I.__bases__
# __bases__ is a class-related attribute available through the class/metaclass attribute lookup mechanisms.

# The Metaclass/Class Dichotomy
    # Python unified classes and types, but this creates some complexity. The author questions whether this unification is worth the extra complications.
# What is the dichotomy?
# Class: creates ordinary instances, such as A() or MyClass().
# Metaclass: creates class objects; Python uses type by default.
type(object)                                 # Fetch object type
type(classname, superclasses, attributedict) # Make a class/type

# 1. type() has two roles
class A:
    pass
x = A()
print(type(x))  # <class '__main__.A'>

# Role 1 — inspect an object's type: pass one argument.
type(x)  # Returns the object's type
# Role 2 — create a class dynamically: pass three arguments.
MyClass = type(
    "MyClass",       # Class name
    (object,),       # Parent classes
    {"x": 10}        # Class attributes
)
print(MyClass.x)  # 10

# And One “object” to Rule Them All
    # Every Python class ultimately inherits from the built-in object class, directly or indirectly.
# 1. object is inherited automatically
class C:
    pass
class D(object):
    pass
print(C.__bases__)  # (<class 'object'>,)
print(D.__bases__)  # (<class 'object'>,)

# 2. object and type have a circular relationship
print(type(C))       # <class 'type'>
print(type(object))  # <class 'type'>
print(type(type))    # <class 'type'>

print(type.__bases__)    # (<class 'object'>,)
print(object.__bases__)  # ()
# type inherits from object.
# object is an instance of type.
# type is also an instance of itself.

# 3. Why is object useful?
# Every class inherits default behavior from object, including methods such as __repr__()
class C:
    pass
x = C()
print(repr(x))  # Default object representation

class C: pass
class D(object): pass
dir(C) == dir(D)
C.__bases__, D.__bases__

type is object
type(type), type(object)
isinstance(type, object), isinstance(object, type)
type.__bases__, object.__bases__

class C: pass   # All classes inherit object defaults
X = C()
X.__repr__

# Advanced Attribute Tools
    # __slots__ declares which instance attributes a class can have.
    # It can prevent accidental attributes and reduce memory usage when creating many instances.

# Slots: Attribute Declarations
# Slot basics
# 1. Declare allowed attributes
class Limiter:
    __slots__ = ['age', 'name', 'job']
person = Limiter()
person.age = 40
person.name = "Alex"
print(person.age)  # 40
person.ape = 1000  # AttributeError

# 2. Attributes must still be assigned before use
person = Limiter()
print(person.age)  # AttributeError: not assigned yet
person.age = 40
print(person.age)  # 40

# 3. Why use __slots__?
# Catch typos: accidental attribute names are rejected.
# Save memory: instances can avoid storing attributes in a separate __dict__.
# Potentially improve speed: speed benefits may be trivial in modern Python.

# You shouldn’t normally use slots
    # Avoid __slots__ unless you have a clear reason, especially a need to save memory when creating a huge number of objects.

# Slots and namespace dictionaries
    # Normally, Python stores an object's instance attributes in __dict__. With __slots__,
    # attributes can be stored without a dictionary, which saves memory but changes how you access and manage them.

# 1. Slots normally remove __dict__
class Person:
    __slots__ = ['name', 'age']
p = Person()
p.name = "Alex"
p.age = 40
print(p.name)       # Alex
print(p.__dict__)   # AttributeError

# 2. getattr(), setattr(), and dir() still work
print(getattr(p, 'name'))  # Alex
setattr(p, 'age', 41)
print(p.age)               # 41
print('name' in dir(p))    # True

# 3. Add __dict__ to allow extra attributes
class Person:
    __slots__ = ['name', '__dict__']
p = Person()
p.name = "Alex"
p.city = "Nukus"    # Allowed: stored in __dict__
print(p.name)       # Alex
print(p.__dict__)   # {'city': 'Nukus'}

# 4. Why listing attributes is tricky
for attr in list(getattr(p, '__dict__', [])) + getattr(p, '__slots__', []):
    print(attr, '=>', getattr(p, attr))


class C:
    __slots__ = ['a', 'b']  # __slots__ means no __dict__ by default

I = C()
I.a = 1
I.a
I.__dict__

getattr(I, 'a')
setattr(I, 'b', 2)      # But getattr() and setattr() still work
I.b
'a' in dir(I)           # And dir() finds slot attributes too
'b' in dir(I)           # Though __dict__ access will fail
I.__dict__

class D:
    __slots__ = ['a', 'b']
    def __init__(self):
        self.d = 4      # Cannot add new names if no __dict__
I = D()

class D:
    __slots__ = ['a', 'b', '__dict__']  # Name __dict__ to include one too
    c = 3                               # Class attrs work normally
    def __init__(self):
        self.d = 4                      # d stored in __dict__, a is a slot
I = D()
I.d
I.c
I.a                                     # All instance attrs undefined until assigned
I.a = 1
I.b = 2

I.__dict__                  # Some objects have both __dict__ and slot names
                            # getattr() can fetch either type of attr
I.__slots__

getattr(I, 'a'), getattr(I, 'c'), getattr(I, 'd')   # Fetches all 3 forms

for attr in list(I.__dict__) + I.__slots__:         # Wrong...
    print(attr, '=>', getattr(I, attr))

for attr in list(getattr(I, '__dict__', [])) + getattr(I, '__slots__', []):
    print(attr, '=>', getattr(I, attr))

# Multiple __slot__ lists in superclasses
    # The main idea is that __slots__ are not automatically combined into one list through inheritance,
    # even though an instance can use slots from multiple classes

class E:
    __slots__ = ['c', 'd']          # Superclass has slots

class D(E):
    __slots__ = ['a', '__dict__']   # But so does its subclass

I = D()                             # The instance gets the union of each
dir(I)
I.a = 1; I.b = 2; I.c = 3           # slots: a, c, __dict__: b
I.a, I.c

# Inspecting just the inherited slots list won’t pick up slots defined higher in a class tree:
E.__slots__                         # But __slots__ not concatenated
D.__slots__
I.__slots__                         # Instance inherits *lowest* __slots__
I.__dict__                          # And has its own attr dict

for attr in list(getattr(I, '__dict__', [])) + getattr(I, '__slots__', []):
    print(attr, '=>', getattr(I, attr))
b => 2                              # Other superclass slots missed!
a => 1
dir(I)                              # But dir() includes all slot names

# Handling slots and other “virtual” attributes generically
    # __dict__ alone cannot show all an object's attributes. Use dir() to find attribute names and getattr() to retrieve their values.
class Slotful:
    __slots__ = ['a', 'b', '__dict__']
    def __init__(self, data):
        self.c = data

I = Slotful(3)
I.a, I.b = 1, 2
I.a, I.b, I.c                               # Normal attribute fetch
I.__dict__                                  # Both __dict__ and slots storage
[x for x in dir(I) if not x.startswith('__')]
I.__dict__['c']                             # __dict__ is only one attr source
getattr(I, 'c'), getattr(I, 'a')            # dir+getattr is broader than __dict__
                                            # applies to slots, properties, descrip
for a in (x for x in dir(I) if not x.startswith('__')):
    print(a, '=>', getattr(I, a))

# Slot usage rules
    # __slots__ saves memory by avoiding an instance __dict__, but to work effectively, it requires careful design across the inheritance hierarchy.
    # 1. The six rules of __slots__
    # Rule	Explanation
    # 1. Subclass has slots, superclass doesn't	The inherited instance dictionary still exists, so memory savings are limited.
    # 2. Superclass has slots, subclass doesn't	The subclass gets an instance dictionary, reducing the memory benefit.
    # 3. Duplicate slot names	A subclass redefining a parent's slot hides the parent's slot descriptor. Avoid this.
    # 4. No class-level defaults	You cannot define a class attribute with the same name as a slot.
    # 5. Multiple inheritance	Two parent classes with nonempty slots generally cause a layout conflict.
    # 6. No __dict__ or weak references by default	Slots restrict arbitrary attributes and weak references unless explicitly enabled.

class C: pass                   # Bullet 1: slots in sub but not super
class D(C): __slots__ = ['a']   # Makes instance dict for non-slots
I = D()                         # But slot name still managed in class
I.a = 1; I.b = 2
I.__dict__
D.__dict__.keys()

class C: __slots__ = ['a']      # Bullet 2: slots in super but not sub
class D(C): pass                # Makes instance dict for non-slots
I = D()                         # But slot name still managed in class
I.a = 1; I.b = 2
I.__dict__
C.__dict__.keys()

class C: __slots__ = ['a']      # Bullet 3: only lowest slot accessible
class D(C): __slots__ = ['a']   # Superclass slot 'a' is pointless

class C: __slots__ = ['a']; a = 99 # Bullet 4: no class-level defaults

class C: __slots__ = ['a']         # Bullet 5: only one nonempty in mixins
class D: __slots__ = ['a']         # Use empty slots or omit in all but one
class E(C, D): pass

# Example impacts of slots: ListTree and mapattrs
    # a class without __slots__, such as ListTree, can preserve an instance's __dict__
    # even when mixed with classes that use slots. This can make tools work,
    # but it may reduce the memory-saving benefit of slots.

class C(ListTree): pass
I = C()                                   # OK: no __slots__ used
print(I)
class C(ListTree): __slots__ = ['a', 'b'] # OK: superclass produces __dict__
I = C()
I.c = 3
print(I)                                  # Displays c at I, a and b at C

# Multiple inheritance
class A: __slots__ = ['a']                # Both OK by bullet 1 above
class B(A, ListTree): pass
print(B())
class A: __slots__ = ['a']
class B(A, ListTree): __slots__ = ['b']   # Displays b at B, a at A
print(B())

# Unsafe approach:
attrs = obj.__dict__
# Safer approach:
attrs = getattr(obj, '__dict__', {})

# To inspect a broader range of attributes:
for name in dir(obj):
    if not name.startswith('__'):
        try:
            print(name, getattr(obj, name))
        except Exception:
            pass

# What about slots speed?
    # __slots__ usually saves memory, but its speed advantage is small and depends on the workload and Python implementation.
%run slots-test.py

# Properties: Attribute Accessors
    # a property allows you to use a method like an ordinary attribute.
    # @property — controls reading an attribute.
    # @age.setter — controls assignment and can validate values.
    # __getattr__ — runs when an attribute isn't found normally.
    # __setattr__ — intercepts every attribute assignment.
    # __slots__ — controls attribute storage and can reduce memory usage

    # Properties control how attributes behave; slots control how attributes are stored.

# Property basics
    # A property lets a class control how an attribute is read, assigned, or deleted, without changing the usual syntax obj.age


class WithOperators:
    def __getattr__(self, name): # On undefined attr
        if name == 'age':
            return 40
        else:
            raise AttributeError(name)
x = WithOperators()
x.age                           # Runs __getattr__
x.name                          # Runs __getattr__

# 1. Creating a property
class WithProperties:
    def getage(self):
        return 40
    age = property(getage)  # (get?, set?, del?, docs?), or @
x = WithProperties()
x.age                       # Runs getage
x.name                      # Normal fetch

# 2. Adding a setter
class WithProperties:
    def getage(self):
        print('get age')
        return 40
    def setage(self, value):
        print('set age:', value)
        self._age = value
    age = property(getage, setage)

x = WithProperties()
x.age                   # Runs getage
x.age = 42              # Runs setage
set age: 42
x._age                  # Normal fetch: no getage call
x.job = 'hacker'        # Normal assign: no setage call
x.job                   # Normal fetch: no getage call

class WithOperators:
    def __getattr__(self, name):           # On undefined attr
        if name == 'age':
            print('get age')
            return 40
        else:
            raise AttributeError(name)
    def __setattr__(self, name, value):    # On all assignments
        print('set:', name, value)
        if name == 'age':
            self.__dict__['_age'] = value  # Or object.__setattr__(self, ...)
        else:
            self.__dict__[name] = value
x = WithOperators()
x.age                           # Runs __getattr__
x.age = 41                      # Runs __setattr__
x._age                          # Defined: no __getattr__ call
x.job = 'coder'                 # Runs __setattr__ again
x.job                           # Defined: no __getattr__ call

# 4. The modern decorator syntax
class WithProperties:
    @property                   # Coding properties with decorators: ahead
    def age(self):              # On instance.age
    @age.setter
    def age(self, value):       # On instance.age = value

# __getattribute__ and Descriptors: Attribute Implementations
    # __getattribute__ intercepts every attribute read, while descriptors control access to specific attributes.
# 1. __getattribute__ versus __getattr__
class Person:
    def __getattribute__(self, name):
        print("Reading:", name)
        return object.__getattribute__(self, name)

p = Person()
p.age = 40
print(p.age)

    # A descriptor is an object that defines methods such as __get__() or __set__() to control access to a particular attribute.
class AgeDesc:
    def __get__(self, instance, owner): return 40
    def __set__(self, instance, value): instance._age = value

class WithDescriptors:
    age = AgeDesc()         # Assign descriptor instance

x = WithDescriptors()
x.age                   # Runs AgeDesc.__get__
x.age = 42              # Runs AgeDesc.__set__
x._age                  # Normal fetch: no AgeDesc call

# Static and Class Methods
    # static methods and class methods. Unlike ordinary instance methods,
    # they do not require a specific object instance in the usual way.
# 1. Ordinary instance method
class Student:
    def greet(self):
        print("Hello!")

s = Student()
s.greet()

# 2. Static method — @staticmethod
class Math:
    @staticmethod
    def add(a, b):
        return a + b
print(Math.add(3, 5))  # 8
m = Math()
print(m.add(4, 6))     # 10

# 3. Class method — @classmethod
class Student:
    school = "Central School"

    @classmethod
    def get_school(cls):
        return cls.school
print(Student.get_school())  # Central School

# Why the Special Methods?
    # sometimes a method needs to work with the class itself rather than with an individual object.
    # 1. The problem: tracking objects
    # 1. The problem: tracking objects
class Student:
    count = 0
    def __init__(self, name):
        self.name = name
        Student.count += 1
s1 = Student("Ali")
s2 = Student("Malika")
s3 = Student("John")
print(Student.count)  # 3

    # 2. Why not use an ordinary function outside the class?
def get_count():
    return Student.count

    # better organized:
class Student:
    count = 0
    def __init__(self, name):
        self.name = name
        Student.count += 1
    @staticmethod
    def get_count():
        return Student.count
s1 = Student("Ali")
s2 = Student("Malika")
print(Student.get_count())  # 2
print(s1.get_count())       # 2
    # 3. Why use a class method instead?
    # A class method receives the class itself through cls
class Student:
    count = 0
    @classmethod
    def get_count(cls):
        return cls.count
class GraduateStudent(Student):
    count = 10
print(Student.get_count())         # 0
print(GraduateStudent.get_count()) # 10

# Plain-Function Methods
    # Regular method → receives self automatically when called through an instance.
    # @staticmethod → receives no automatic self or cls.
    # @classmethod → receives the class as its first argument, conventionally named cls.
from hack1 import Hack
a, b, c = Hack(), Hack(), Hack() # Make three instances

Hack.printNumInstances()        # Okay to call from instance – only
a.printNumInstances()

# Static Method Alternatives
What are the disadvantages of the external function?
    # `1. The function is separated from the class.
    # The function processes Hack data, but it isn't structurally part of the Hack class.
    # 2. It adds another name to the module.
    # printNumInstances() exists in the module's namespace rather than the class's namespace.
    # 3. Inheritance cannot customize it naturally.`
import hack2 as hack
a = hack.Hack()
b = hack.Hack()
c = hack.Hack()
hack.printNumInstances()    # But function may be too far removed
hack.Hack.numInstances

# Using Static and Class Methods
    # Python has three types of methods. They differ in what Python automatically passes as the first argument.
    # 1. Instance method — receives self. Use it when: The method needs to access or modify a particular object's data.
    # 2. Static method — receives no automatic argument. Use it when: A function logically belongs to the class but doesn't need instance or class data.
    # 3. Class method — receives cls. Use it when: The method needs to work with the class itself, often to create instances or access class-level data.

from allmethods import Methods  # Normal instance methods
obj = Methods()                 # Callable through instance or class
obj.imeth(1)                    # Becomes imeth(obj, 1)
Methods.imeth(obj, 2)

Methods.smeth(3)    # Static method: call through class
                    # No instance passed or expected
obj.smeth(4)        # Static method: call through instance
                    # Instance not passed – requires staticmethod

Methods.cmeth(5)    # Class method: call through class
obj.cmeth(6)        # Class method: call through instance
                    # Becomes cmeth(Methods, 6)

# Counting Instances with Static Methods
    # A static method can access and display class-level data without receiving self or cls automatically.
from hack_static import Hack
a, b, c = Hack(), Hack(), Hack()
Hack.printNumInstances()    # Call as simple function
a.printNumInstances()       # Instance argument not passed

class Sub(Hack):
    def printNumInstances():        # Override a static method
        print('Extra stuff...')     # But call back to original
        Hack.printNumInstances()
    printNumInstances = staticmethod(printNumInstances)

a, b = Sub(), Sub()
a.printNumInstances()       # Call from subclass instance
Sub.printNumInstances()     # Call from subclass itself
Hack.printNumInstances()    # Call original version

class Other(Hack): pass     # Inherit static method verbatim
c = Other()
c.printNumInstances()

# Counting Instances with Class Methods
    # A class method works similarly to a static method, but Python automatically passes the calling class as the first argument (cls).
from hack_class import Hack
a, b = Hack(), Hack()
a.printNumInstances()       # Passes class to first argument
Hack.printNumInstances()    # Also passes class to first argument

from hack_class2 import Hack, Sub, Other
x = Sub()
y = Hack()
x.printNumInstances()       # Call from subclass instance
Sub.printNumInstances()     # Call from subclass itself
y.printNumInstances()       # Call from superclass instance

z = Other()                 # Call from lower sub's instance
z.printNumInstances()

# Counting instances per class with class methods
    # Class methods allow each class in an inheritance hierarchy to maintain its own instance counter.
    # Instance method(self)-Instance-specific data
    # Static method(Nothing)-Data or operations local to a class
    # Class method(cls)-Data that differs across classes in an inheritance hierarchy
from hack_class3 import Hack, Sub, Other
x = Hack()
y1, y2 = Sub(), Sub()
z1, z2, z3 = Other(), Other(), Other()
x.numInstances, y1.numInstances, z1.numInstances        # Per-class data!
Hack.numInstances, Sub.numInstances, Other.numInstances

# Decorators and Metaclasses
    # Decorators let you add extra functionality to functions, methods, and classes without rewriting their original code.

# Function Decorator Basics
    # Decoration takes a function, passes it to another function (the decorator), and reassigns the returned result to the original function name.

    # Instance method-Needs an object's data-account.deposit(500)
    # Class method-Needs class-level data or the calling class- BankAccount.from_file()
    # Static method-Needs neither object nor class data-BankAccount.validate_amount(500) (# Independent calculation)
from hack_static_deco import Hack
a, b, c = Hack(), Hack(), Hack()
Hack.printNumInstances() # Calls from classes and instances work
a.printNumInstances()

from alldecorators import Methods
obj = Methods()
obj.imeth(1)
obj.smeth(2)
obj.cmeth(3)
obj.name

# A First Look at User-Defined Function Decorators
    # Function-based decorators (def) are generally the preferred approach for most everyday situations
%run tracer1.py

# A First Look at Class Decorators and Metaclasses
@decorator
class Hack:
    pass
# Hack = decorator(Hack)

# ------------
def count(aClass):
    aClass.numInstances = 0
    return aClass

@count
class Hack:
    pass
# Hack = count(Hack)

@count
class Sub(Hack):
    pass
# Sub = count(Sub)

print(Hack.numInstances)  # 0
print(Sub.numInstances)   # 0

# ------------
def count(obj):
    obj.numInstances = 0
    return obj

@count
def hack():  # Use it with a function:
    pass
@count
class Hack:  # Or with a class:
    pass

print(hack.numInstances)  # 0
print(Hack.numInstances)  # 0

    # Instead of modifying the original class, a decorator can return a completely different class that acts as a proxy
# ------------
def decorator(cls):                     # On @ decoration
    class Proxy:
        def __init__(self, *args):      # On instance creation: make a cls
            self.wrapped = cls(*args)
        def __getattr__(self, name):    # On attribute fetch: extra ops here
            return getattr(self.wrapped, name)
    return Proxy

@decorator
class C:                              # Like C = decorator(C)
    def __init__(self, name):
        self.name = name
    def hello(self):
        print("Hello", self.name)
X = C("Ali")                         # Makes a Proxy that wraps a C, and catches later X.attr

# ------------
# Python uses the built-in metaclass "type".
# A metaclass is essentially a class that controls how other classes are created.
class Hack:
    pass

type(Hack)  # <class 'type'>

# You can define your own metaclass:
class Meta(type):
    def __new__(meta, classname, supers, classdict):
        print("Creating class:", classname)
        return type.__new__(
            meta, classname, supers, classdict
        )

class Hack(metaclass=Meta):  # Creating class: Hack
    pass

# The super Function
    # super() allows a class to call a method from the next class in the inheritance lookup order without explicitly naming that parent class
    # super() is powerful.Need to understand MRO
class A:
    def hello(self):
        print("Hello from A")

class B(A):
    def hello(self):
        print("Hello from B")
        A.hello(self)   # super().hello()

# The super Basics
    # basic difference between explicitly calling a superclass method and using super()
# 1. Explicit superclass call (without super())
class C:
    def act(self):
        print('hack')

class D(C):
    def act(self):
        C.act(self)     # Name superclass explicitly, pass self
        print('code')
I = D()
I.act()

# 2. Using super()
class C:
    def act(self):
        print('hack')

class D(C):
    def act(self):
        super().act() # Reference superclass implicitly, omit self
        print('code')
I = D()
I.act()

# The super Details
# A “magic” proxy
    # what happens internally when you call super() and why the book considers its behavior unusual.
    # 1. Why is super() unusual?
class C:
    def act(self):
        print('hack')

class D(C):
    def act(self):
        super().act()
        print('code')
    # Inside super().act(), we did not explicitly provide:
    # - The current class (D).
    # - The superclass (C).
    # - The current instance (self).

    # 2. A "magic" proxy
    # super() returns a special proxy object.
    # This proxy searches for an attribute or method in the MRO, starting after the class where the super() call is made.
class C:
    def act(self):
        print('hack')

class D(C):
    def act(self):
        proxy = super()
        proxy.act()
        print('code')
    # 3. How does Python know the class and instance?
    # A. Runtime call stack
    # B. __class__ variable
    # 4. Two equivalent forms of super()
super() #and
super(D, self)
    # 5. Why doesn't super() work outside a method?
super(D, obj)  #the two-argument form is possible outside a method
    # 6. Understanding the proxy object
class E(C):
    def method(self):
        proxy = super()
        return proxy
prx = E().method()
prx.act()       # Find act on MRO past hidden E, bind with hidden self, call (!)


# Attribute-fetch algorithm
    # exactly how super() finds an attribute, especially when a class uses multiple inheritance.
    # The key point is that super() does not simply search the direct parent class. It follows the MRO of the actual instance's class
# 1. Single inheritance: a straightforward case
class C:
    pass
class E(C):
    pass
E.__mro__ #(<class '__main__.E'>, <class '__main__.C'>, <class 'object'>)
# the immediate superclass using
E().__class__.__bases__[0]  #<class '__main__.C'>

#-----------------
class A:
    def action(self):
        print("A")
class B:
    def action(self):
        print("B")
class C(A):
    def action(self):
        print("C")
        super().action()
class D(C, B):
    pass

obj = D()
obj.action()
>>> D.__mro__ #(D, C, A, B, object)

# Universal deployment
    # super() does not necessarily call the method of the parent class you expect when multiple inheritance is involved.

    # 1. Universal deployment – single inheritance
class A:
    def act(self):
        print('A')
class B:
    def act(self):
        print('B')
class C(B):                     # C inherits from B
    def act(self):
        super().act()            # Call the next act() method in the MRO (B)
C().act()                        # Create C instance and call act() # Output: B

    # 2. Multiple inheritance – B comes first
class A:
    def act(self):
        print('A')
class B:
    def act(self):
        print('B')
class C(B, A):                  # C inherits from B first, then A
    def act(self):
        super().act()           # Finds the first act() after C in the MRO (B)
C().act()                       # Output: B
print(C.__mro__)                # Display the Method Resolution Order  # (<class 'C'>, <class 'B'>, <class 'A'>, <class 'object'>)

    # 3. Multiple inheritance – A comes first
class A:
    def act(self):
        print('A')
class B:
    def act(self):
        print('B')
class C(A, B):                  # Change parent order: A comes before B
    def act(self):
        super().act()           # Finds A.act() first according to the MRO
C().act()                       # Output: A
print(C.__mro__)                # Display the new MRO # (<class 'C'>, <class 'A'>, <class 'B'>, <class 'object'>)

    # 4. Explicit class calls – execute both parent methods
class A:
    def act(self):
        print('A')
class B:
    def act(self):
        print('B')
class C(A, B):                  # C inherits from both A and B
    def act(self):
        A.act(self)              # Explicitly call A's method, passing C instance
        B.act(self)              # Explicitly call B's method, passing C instance
C().act()                       # Output: # A # B

    # 5. Universal deployment – every class uses super()
    # Universal deployment means every participating class must use super() consistently if you want the method call to propagate through the entire MRO
class Root:
    def act(self):
        print('Root')            # Final method in the cooperative call chain
class A(Root):
    def act(self):
        print('A')
        super().act()            # Continue to the next class in the MRO (B)
class B(Root):
    def act(self):
        print('B')
        super().act()            # Continue to the next class in the MRO (Root)
class C(A, B):
    def act(self):
        print('C')
        super().act()            # Continue to the next class in the MRO (A)

C().act()                       # Output: # C # A
print(C.__mro__)                # Check the actual method resolution order # C -> A -> B -> Root -> object

    # 6. What happens if one class does not use super()?
class Root:
    def act(self):
        print('Root')
class A(Root):
    def act(self):
        print('A')
        # No super().act() here!
        # The cooperative method chain stops at A
class B(Root):
    def act(self):
        print('B')
        super().act()            # This would continue to Root if B were reached
class C(A, B):
    def act(self):
        print('C')
        super().act()            # Calls A.act() according to the MRO
C().act() # Output:,# C,# A. # B.act() and Root.act() are never executed because A.act() does not call super()

# Call-chain anchors
    # Cooperative method dispatch means that each class uses super() to pass the method call to the next class in the MRO.
    # Each class runs its method once, following the MRO order.
    # However, there is a problem: the call chain needs an endpoint (anchor) to stop the propagation.
# Example 1. Without an anchor
class A:
    def act(self):
        print('A')
        super().act()       # Add a super here too
class B:
    def act(self):
        print('B')
        super().act()       # Add a super here too
class C(B, A):
    def act(self):
        super().act()       # Hope this winds up running all methods
C().act()  # A # AttributeError: 'super' object has no attribute 'act'
    # Why does this happen? # First, inspect the MRO:
I = C()
I.__class__.__mro__  # (<class 'C'>, <class 'B'>, <class 'A'>, <class 'object'>)
    # The call sequence is:
    # 1. C calls super().act() → selects B.act().
    # 2. B calls super().act() → selects A.act().
    # 3. A calls super().act() → looks for act() in object.
    # 4. object has no act() method → AttributeError.
    # There is no call-chain anchor: no final class that catches the call and stops the propagation.

# Example 2. Adding an anchor class X
# We can create a class X that defines act() but does not call super().
class X:                     # Code a bogus class, just to appease super
    def act(self):
        print('anchor')

class A: ...                 # Same as before
class B: ...                 # Same as before

class C(B, A, X):             # Add a final anchor, just to appease super
    def act(self):
        super().act()
C().act()  # B # A # anchor

[c.__name__ for c in C().__class__.__mro__]
# ['C', 'B', 'A', 'X', 'object']

# Example 3. Anchor as a common superclass
# Instead of adding X directly to C, we can make it a common superclass of both A and B.
class X: ...                 # Same anchor class
class A(X): ...              # Same method as before
class B(X): ...              # Same method as before
class C(B, A): ...           # Same method as before
C().act() #B # A # anchor
[c.__name__ for c in C().__class__.__mro__]
# ['C', 'B', 'A', 'X', 'object']

# Same argument lists
    # that object can act as a free anchor for constructor calls because object.__init__() exists and does not call super() again.
# Example 1. Without constructor arguments
class A:
    def __init__(self):
        print('A')
        super().__init__()       # Propagate constructor calls
class B:
    def __init__(self):
        print('B')
        super().__init__()       # Propagate constructor calls
class C(B, A):
    def __init__(self):
        super().__init__()       # Assume object anchors constructor chain
I = C() # B # A
# MRO: C → B → A → object
    # Call sequence:
    # 1. C calls B.__init__().
    # 2. B prints B, then calls A.__init__().
    # 3. A prints A, then calls object.__init__().
    # 4. object.__init__() accepts the call and ends the chain

# Example 2. With constructor arguments
# add a name argument to every constructor.
class A:
    def __init__(self, name):    # Add an argument to the method
        print('A')
        super().__init__(name)
class B:
    def __init__(self, name):
        print('B')
        super().__init__(name)
class C(B, A):
    def __init__(self, name):
        super().__init__(name)   # But object's arguments list differs
I = C('Pat')    # B # A # TypeError: object.__init__() takes exactly one argument (the instance to initialize)

# Why does it fail?
# The call sequence is:
# 1. C passes 'Pat' to B.
# 2. B passes 'Pat' to A.
# 3. A passes 'Pat' to object.__init__().
# 4. But object.__init__() accepts only self, not name.

# Noncalls and operator overloading
    # super() can access not only methods but also class data attributes and bound methods.
class C:
    attr1 = 'hack'             # super also fetches data attributes
    def attr2(self):           # And returns bound methods sans calls
        return 'code'
class D(C):
    def act(self):
        return super().attr1, super().attr2
I = D()
I.act()  # ('hack', <bound method C.attr2 of <__main__.D object ...>>)
    # There are two results:
    # - super().attr1 → retrieves the class attribute 'hack'.
    # - super().attr2 → retrieves a bound method, but does not execute it.

I.act()[1]() #execute the method #

# 2. Operator overloading
# The second limitation concerns special methods such as __getitem__()
class C:
    def __getitem__(self, ix):       # Indexing overload method
        print('C index')
class D(C):
    def __getitem__(self, ix):       # Redefine to extend here
        print('D index')
        C.__getitem__(self, ix)      # Explicit call form works
        super().__getitem__(ix)      # Direct name calls work too
        super()[ix]                  # But operators do not! (__getattribute__)
I = C()
I[99]       # C index
I = D()
I[99]       # TypeError: 'super' object is not subscriptable

    # The reason is that Python handles operator overloading differently.
    # - super().__getitem__(ix) explicitly requests the __getitem__ attribute from the super proxy.
    # - super()[ix] uses the indexing operator. Python looks for the special method on the class of
    # the object rather than using the proxy's normal attribute lookup (__getattribute__).
    # Therefore, super() cannot intercept this automatic operator invocation.

# The super Wrap-Up
    # 1. Where super() is useful
    # - Single inheritance: super() allows you to refer to a parent class without explicitly writing its name.
    # - Multiple inheritance: super() can implement cooperative method dispatch, passing method calls through the MRO.
    # - Diamond inheritance: A properly designed call chain visits each superclass only once
    # 2. Other limitations
    # super() depends on the already complex MRO.
    # Code that works with single inheritance may behave differently when multiple inheritance is introduced.
    # In multiple inheritance, super() may select a sibling class rather than a superclass.
    # super() introduces additional special rules for attribute inheritance.

# Class Gotchas
    # Most problems with classes are related to namespaces, because classes are essentially namespaces with some additional features.

# Changing Class Attributes Can Have Side Effects
    # classes and instances are mutable objects. This means we can change their attributes after they have been created.
    # 1. Class attributes are shared by instances
class X:
    a = 1                 # Class attribute
I = X()
I.a                       # Inherited by instance: 1
X.a                       # Accessible through class: 1
    # 2. Changing a class attribute
X.a = 2                   # May change more than X
I.a                       # I changes too: 2
J = X()                   # New instance
J.a                       # 2
    # Important distinction:
J.a = 3
    # Now J gets its own attribute a.
    # - J.a → 3
    # - I.a → 2
    # - X.a → 2
    # 3. Using classes as namespaces without instances
class X: pass             # Make a few attribute namespaces # The classes work like fileless modules
class Y: pass

X.a = 1                   # Use class attributes as variables
X.b = 2                   # No instances anywhere to be found
X.c = 3
Y.a = X.a + X.b + X.c

for X.i in range(Y.a):
    print(X.i)            # Prints 0..5
    # 4. Using instances instead of classes
    # changing class attributes in classes written by other programmers
    # can be dangerous because those attributes may be important to the class's internal behavior.
# C struct (a record containing fields), it may be better to use instances:
class Record: pass
X = Record()
X.name = 'pat'
X.job = 'Pizza maker'

# - Record defines an empty class.
# - X is an instance of Record.
# - name and job are attributes belonging only to X.

# Changing Mutable Class Attributes Can Have Side Effects, Too
    # Mutable class attributes are shared by all instances, so
    # changing their contents through one instance affects all other instances.
# 1. Class attribute vs instance attribute
class C:
    shared = []             # Class attribute
    def __init__(self):
        self.perobj = []    # Instance attribute
x, y = C(), C()             # Two instances
y.shared, y.perobj          # Implicitly share class attrs # ([], [])
    # shared belongs to the class and is shared by instances.
    # perobj belongs separately to each instance
# 2. Changing mutable attributes
x.shared.append('hack')    # Impacts y's view too!
x.perobj.append('code')    # Impacts x's data only
x.shared, x.perobj         # (['hack'], ['code'])

# Now check instance y:
y.shared, y.perobj         # (['hack'], [])
# And the class itself:
C.shared                    # ['hack']
    # x.shared and y.shared refer to the same list stored in class C.
    # x.perobj and y.perobj refer to two different lists.
    # Therefore, modifying x.shared also changes what y.shared sees, but modifying x.perobj does not affect y.perobj.
# 3. Important distinction: append() vs assignment
# Case 1: Modify the shared list in place
x.shared.append('hack')    # Changes shared object attached to class - in place
# Case 2: Assign a new value
x.shared = 'hack'          # Changed or creates instance attribute attached to x.
                           # This creates an instance attribute shared for x, rather than modifying the class attribute.

# Multiple Inheritance: Order Matters
    # the order of parent classes in the class definition matters because
    # Python searches inherited attributes from left to right according to the MRO.
# 1. Order determines which method is inherited
class ListTree:
    def __str__(self): ...
class Super:
    def __str__(self): ...

class Sub(ListTree, Super):  # Get ListTree's __str__ by listing it first
    pass
x = Sub()                    # Inheritance searches ListTree before Super

class Sub(Super, ListTree): # Now Python finds Super.__str__() first.
    pass

# 2. What if we need methods from both parents?
class ListTree:
    def __str__(self): ...
    def other(self): ...

class Super:
    def __str__(self): ...
    def other(self): ...

    # We want:
    # - ListTree.__str__()
    # - Super.other()

    # Listing ListTree first gives us its versions of both methods by default. We need to explicitly select Super.other():
class Sub(ListTree, Super):  # Get ListTree's __str__ by listing it first
    other = Super.other      # But explicitly pick Super's version of other
    def __init__(self):
        ...
x = Sub()                    # Inheritance searches Sub before ListTree/Super
    # creates Sub.other, which refers to the method defined in Super.
    # Because Python searches Sub before its parent classes, it finds Sub.other first and does not continue to ListTree.other.
# 3. The reverse approach
# We can also list Super first and explicitly select ListTree.__str__():
class Sub(Super, ListTree):  # Get Super's other by order
    __str__ = ListTree.__str__  # Explicitly pick ListTree.__str__
    # other() comes from Super because it is listed first.
    # __str__() comes from ListTree because we explicitly assign it.
    # The author considers multiple inheritance an advanced feature that should be used carefully.
    # The problem is that the meaning of an attribute can depend on the order in which parent classes are listed,
    # even in a subclass far down the inheritance tree.
# 5. Using pseudoprivate attributes to avoid conflicts
# pseudoprivate __X naming feature.
class ListTree:
    def __str__(self): ...
    def other(self): ...
# can be written
class ListTree:
    def __str__(self): ...  # The double underscore helps localize the method name to ListTree,
    def __other(self): ...  # reducing the possibility of conflicts with similarly named methods in other classes.

# Scopes in Methods and Classes
    # Python's LEGB scope lookup rule works when classes and methods are nested inside functions.
    # methods can access variables from enclosing functions, but they cannot directly access the local namespace of their enclosing class.
# 1. Nested class inside a function
def generate():
    class Hack:                  # Hack is a name in generate's local scope
        count = 1
        def method(self):
            print(Hack.count)    # Visible in generate's scope, per LEGB rule (E)
    return Hack()
generate().method() # 1
    # Why does this work?
    # - generate() defines the class Hack.
    # - Hack is a local name in generate().
    # - method() is nested inside the class, which is nested inside generate().
    # - The method can find Hack in the enclosing function's scope (E in LEGB).
# 2. Methods cannot access the enclosing class's local scope
# The author emphasizes an important distinction.

# Inside method(), we cannot simply write:
print(count)       # Not accessible directly
# Instead, we must use:
print(self.count)  # Access through the instance
print(Hack.count)  # Access through the class name
    # Because class namespaces are not enclosing function scopes for methods.
    # Methods can access enclosing function scopes, but not the local scope of the class definition.

# 3. Moving the class to the module level
# The author shows that we can avoid nesting the class inside the function
def generate():
    return Hack()
class Hack:                       # Define at top level of module
    count = 1
    def method(self):
        print(Hack.count)         # Found in global scope (enclosing module)
generate().method()               # 1
    # Now Hack is defined at the module level.
    # Both generate() and method() can find Hack through the module's global namespace.
# 4. Using nested classes with closures
# Nesting classes - it is useful when we want a class or its methods to retain access to variables from an enclosing function.

def generate(label):              # Returns a class instead of an instance
    class Hack:
        count = 1
        def method(self):
            print(f'{label}={Hack.count}')
    return Hack

aclass = generate('Gotchas')
I = aclass()
I.method()                         # Gotchas=1
# Here, generate() returns the class itself, not an instance.
# The method can access:
# - label — from the enclosing function's scope.
# - Hack.count — through the class name.
# - self — the current instance.
# The variable label remains available to the method even after generate() has finished executing. This is an example of a closure.

# Miscellaneous Class Gotchas
# 1. Choose per-instance or class storage wisely.
    # Class attributes are shared by all instances.
    # Instance attributes (self.attribute) belong to individual instances.
# 2. You usually want to call superclass constructors.
    # Python automatically calls only the first __init__() it finds through inheritance. It does not automatically execute all superclass constructors.
# 3. Stay tuned for a fix for __getattr__ and built-ins
    # When using __getattr__() to delegate attribute access to another object, some built-in operations may fail.
    # This happens because built-in operations that invoke special methods (such as operator-overloading methods) do not use generic attribute interception.

# “Overwrapping-itis”
# OOP abstraction improves code reuse, polymorphism, and encapsulation, but excessive abstraction makes code unnecessarily complicated.
    # Author's recommendations:
    # - Avoid unnecessary class layers.
    # - Keep inheritance hierarchies short.
    # - Make class interfaces intuitive.
    # - Use abstraction when it provides real benefits.

SyntaxError: invalid syntax (2247513062.py, line 267)